# REMNET Stage 3 — REMNET Ensemble (Stage 2 seed-matched checkpoints)

**Sections 3–4: frozen-checkpoint original-test verification and direct external-dataset inference.**

This is the standalone ResNet50 Stage 3 notebook with **only the model implementation changed** to the Stage 2 REMNET ensemble. REMNET combines **MobileNetV2 + ResNet50 + EfficientNet-Lite0**, each loaded from its *seed-matched* Stage 2 REMNET checkpoint, using the **arithmetic mean of three per-class softmax vectors**. Architecture and filenames follow `Remnet_Accu_s2(3).ipynb`.

The original locked Stage 2 test has **463 images** and the same verified external manifest has **4,931 images**. Both use **RGB → Resize(32,32) → ToTensor → Normalize(0.5,0.5,0.5)**. No model is trained, adapted, or selected using D2. All five seeds are evaluated. No annotation audit is performed.


## 1. Configuration and inputs

Uses the same attached Kaggle dataset, manifests and checkpoint root as the original ResNet50 Stage 3 notebook. The only checkpoint change is to `Section3/Classification_checkpoints/Remnet/` (15 constituents, three per seed), **not** `Resnet_50/`. Reproduces the Stage 2 REMNET equal-weight probability ensemble.


In [1]:
from __future__ import annotations
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, traceback
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone

# Kaggle input location confirmed by the user; adjust ONLY if Kaggle moves the dataset.
INPUT_ROOT = Path('/kaggle/input/datasets/akash264208/checkpoint-manifist')
OUTPUT = Path('/kaggle/working/REMNET_Stage3_Ensemble_External_Validation')
OUTPUT.mkdir(parents=True, exist_ok=True)
SEEDS = (7, 49, 112, 345, 476)
MODEL_NAME = 'REMNET'
CONSTITUENTS = ('MobileNetV2','ResNet50','EfficientNet-Lite0')
CLASSES = ('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
EXPECTED_SPLIT_COUNTS = {'Training':1956,'Validation':487,'Testing':463}
EXPECTED_EXTERNAL_COUNT = 4931
EXPECTED_STAGE2_SPLIT_SHA256 = '9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792'
EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT = '0761bdca0f92aea4efd8e5945e2954bf3d6958f41644e8699766839047ddd9be'
EXPECTED_V2_VERSION = 12
BATCH_SIZE = 32

# Optional genuine historical Stage 2 REMNET ensemble prediction files.
# Each file, when present, must have source_path and predicted_class_id columns,
# and preferably all six prob_CLASS columns. They must come from Stage 2.
# Set None to discover files automatically in the attached private Kaggle input.
HISTORICAL_REFERENCE_DIR = None
REQUIRE_HISTORICAL_REFERENCE = False  # Set True to STOP external evaluation if references are absent.

assert INPUT_ROOT.is_dir(), f'Input dataset was not attached: {INPUT_ROOT}'
print('Kaggle input:', INPUT_ROOT)
print('Output directory:', OUTPUT)
print('Seeds:', SEEDS)
print('Frozen three-constituent REMNET ensemble; no training or fine-tuning.')

Kaggle input: /kaggle/input/datasets/akash264208/checkpoint-manifist
Output directory: /kaggle/working/REMNET_Stage3_Ensemble_External_Validation
Seeds: (7, 49, 112, 345, 476)
Frozen three-constituent REMNET ensemble; no training or fine-tuning.


## 2. Dependencies and strict ensemble-checkpoint inventory

Load exactly 15 Stage 2 REMNET constituent checkpoints from the REMNET folder: `MobileNetV2_seed{seed}.pt`, `ResNet50_seed{seed}.pt` and `EfficientNet-Lite0_seed{seed}.pt` (five seeds). The other standalone ResNet50 checkpoints are **not used**. Stage 2 architectures: torchvision MobileNetV2, torchvision ResNet50, and timm EfficientNet-Lite0.


In [2]:
# Load dependencies without downloading pretrained weights.
for name in ('kagglehub', 'timm'):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', name])
import kagglehub, timm, re, numpy as np, pandas as pd, torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU inference selected')
print('torch:', torch.__version__, 'timm:', timm.__version__)

def only_directory_with_name(root, name):
    matches = sorted(p for p in root.rglob('*') if p.is_dir() and p.name.casefold() == name.casefold())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one directory {name!r} under {root}; found {matches}')
    return matches[0]

def unique_by_filename(root, filename):
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r} in {root}: {matches}')
    return matches[0]

CKPT_ROOT = only_directory_with_name(INPUT_ROOT, 'Classification_checkpoints')
REMNET_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Remnet')
STANDALONE_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Resnet_50')
MANIFEST_ROOT = only_directory_with_name(INPUT_ROOT, 'manifist')
RESULTS_ROOT = only_directory_with_name(INPUT_ROOT, 'per_seed_performance_summary')
SPLIT_MANIFEST = unique_by_filename(MANIFEST_ROOT, 'REMNET_Stage2_Original_Split_Manifest.csv')
EXTERNAL_MANIFEST = unique_by_filename(MANIFEST_ROOT, 'Section2_Final_Automated_Screened_Six_Class_Manifest.csv')
CLASS_MAPPING = unique_by_filename(MANIFEST_ROOT, 'Section2_Six_Class_Mapping.csv')

CHECKPOINTS = {}
for seed in SEEDS:
    CHECKPOINTS[seed] = {}
    for name in CONSTITUENTS:
        # Exact original Stage 2 filenames, with optional Kaggle duplicate suffix (1).
        pattern = re.compile(rf'{re.escape(name)}_seed{seed}(?:\(\d+\))?\.pt', re.IGNORECASE)
        found = sorted(p for p in REMNET_CKPT_DIR.rglob('*.pt') if pattern.fullmatch(p.name))
        if len(found) != 1:
            raise RuntimeError(f'Expected one REMNET checkpoint for {name} seed {seed}, found: {found}')
        CHECKPOINTS[seed][name] = found[0]

all_pts = sorted(CKPT_ROOT.rglob('*.pt'))
remnet_pts = sorted(REMNET_CKPT_DIR.rglob('*.pt'))
assert len(remnet_pts) == 15, f'Expected 15 REMNET constituent checkpoints, found {len(remnet_pts)}'
assert len({p for group in CHECKPOINTS.values() for p in group.values()}) == 15
assert len(all_pts) == 30, f'Expected 30 total Stage 2 checkpoints, found {len(all_pts)}'
assert not any(p.is_relative_to(STANDALONE_CKPT_DIR) for group in CHECKPOINTS.values() for p in group.values())

# Stage 2 REMNET-only historical references; do not use standalone ResNet50 reports.
HISTORICAL_PER_SEED = next(iter(sorted(RESULTS_ROOT.rglob('REMNET_Performance_Per_Seed.csv'))), None)
HISTORICAL_SUMMARY = next(iter(sorted(RESULTS_ROOT.rglob('REMNET_5_Seed_Performance_Summary.csv'))), None)
print('Total .pt checkpoints:', len(all_pts), '| REMNET constituent checkpoints:', len(remnet_pts))
for seed, group in CHECKPOINTS.items():
    for name, path in group.items(): print('REMNET', seed, name, ':', path)
for path in [SPLIT_MANIFEST, EXTERNAL_MANIFEST, CLASS_MAPPING]: print('CSV:', path)
print('Historical REMNET per-seed:', HISTORICAL_PER_SEED or 'not present')
print('Historical REMNET summary:', HISTORICAL_SUMMARY or 'not present')


GPU: Tesla T4
torch: 2.11.0+cu130 timm: 1.0.30
Total .pt checkpoints: 30 | REMNET constituent checkpoints: 15
REMNET 7 MobileNetV2 : /kaggle/input/datasets/akash264208/checkpoint-manifist/Section3/Classification_checkpoints/Remnet/MobileNetV2_seed7.pt
REMNET 7 ResNet50 : /kaggle/input/datasets/akash264208/checkpoint-manifist/Section3/Classification_checkpoints/Remnet/ResNet50_seed7.pt
REMNET 7 EfficientNet-Lite0 : /kaggle/input/datasets/akash264208/checkpoint-manifist/Section3/Classification_checkpoints/Remnet/EfficientNet-Lite0_seed7.pt
REMNET 49 MobileNetV2 : /kaggle/input/datasets/akash264208/checkpoint-manifist/Section3/Classification_checkpoints/Remnet/MobileNetV2_seed49.pt
REMNET 49 ResNet50 : /kaggle/input/datasets/akash264208/checkpoint-manifist/Section3/Classification_checkpoints/Remnet/ResNet50_seed49.pt
REMNET 49 EfficientNet-Lite0 : /kaggle/input/datasets/akash264208/checkpoint-manifist/Section3/Classification_checkpoints/Remnet/EfficientNet-Lite0_seed49.pt
REMNET 112 Mobil

## 3. Validate the locked manifests and six-class mapping

The Stage 2 split is checked against its source notebook’s canonical SHA-256, which is also stored in every standalone checkpoint. The external cohort uses the canonical **Section 2 fingerprint** (relative path + image hash + output index), preventing accidental changes or use of the 7,463-image complete audit list.

In [3]:
def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for b in iter(lambda:stream.read(chunk_size),b''):
            h.update(b)
    return h.hexdigest()

def canonical_split_sha(df):
    # Exactly matches the Stage 2 ensemble notebook: sort split + source_path, CSV, SHA-256.
    canonical = df[['split','source_path','class','sha256']].sort_values(
        ['split','source_path']).reset_index(drop=True)
    return hashlib.sha256(canonical.to_csv(index=False).encode('utf-8')).hexdigest()

def canonical_external_fingerprint(df):
    h = hashlib.sha256()
    for row in df.sort_values('external_relative_path').itertuples(index=False):
        h.update((row.external_relative_path+'\t'+row.sha256+'\t'+
                  str(int(row.output_index))+'\n').encode('utf-8'))
    return h.hexdigest()

stage2_df = pd.read_csv(SPLIT_MANIFEST, dtype=str, keep_default_na=False)
external_df = pd.read_csv(EXTERNAL_MANIFEST, dtype=str, keep_default_na=False)
mapping_df = pd.read_csv(CLASS_MAPPING, dtype=str, keep_default_na=False)

assert set(['split','source_path','class','sha256']) <= set(stage2_df.columns)
assert set(['external_relative_path','class','output_index','sha256','final_status']) <= set(external_df.columns)
assert set(['external_folder_class','stage2_trained_class','stage2_output_index']) <= set(mapping_df.columns)

assert len(stage2_df)==2906 and stage2_df['split'].value_counts().to_dict()==EXPECTED_SPLIT_COUNTS
assert len(external_df)==EXPECTED_EXTERNAL_COUNT, 'Unexpected external manifest count (should be 4931).'
assert stage2_df['sha256'].nunique()==2906, 'Duplicate hashes in Stage 2 manifest.'
assert external_df['sha256'].nunique()==EXPECTED_EXTERNAL_COUNT, 'Duplicate hashes in external manifest.'
assert external_df['external_relative_path'].nunique()==EXPECTED_EXTERNAL_COUNT
assert not set(stage2_df.sha256).intersection(set(external_df.sha256)), 'External cohort contains exact Stage 2 overlaps!'
assert set(stage2_df['class']) == set(CLASSES)
assert set(external_df['class']) == set(CLASSES)
assert external_df['final_status'].eq('INCLUDE_AUTOMATED_SCREENED').all(), 'Noneligible images in frozen manifest.'
assert set(mapping_df['external_folder_class'])==set(CLASSES) and len(mapping_df)==6
for item in mapping_df.itertuples(index=False):
    assert item.external_folder_class == item.stage2_trained_class
    assert int(item.stage2_output_index)==CLASS_TO_ID[item.external_folder_class]
    if 'class_mapping_verified' in mapping_df.columns:
        assert str(item.class_mapping_verified).lower()=='true'
# Validate model index explicitly; 'class' is a reserved Python keyword.
assert all(int(i)==CLASS_TO_ID[c] for c,i in zip(external_df['class'],external_df['output_index']))

SPLIT_FINGERPRINT=canonical_split_sha(stage2_df)
EXTERNAL_FINGERPRINT=canonical_external_fingerprint(external_df)
assert SPLIT_FINGERPRINT == EXPECTED_STAGE2_SPLIT_SHA256, (SPLIT_FINGERPRINT,'Stage 2 split mismatch')
assert EXTERNAL_FINGERPRINT == EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT, (EXTERNAL_FINGERPRINT,'External cohort fingerprint mismatch')
print('Stage 2 manifest verified: 1956 / 487 / 463. SHA-256:',SPLIT_FINGERPRINT)
print('Stage 3 frozen manifest verified:',len(external_df),'Fingerprint:',EXTERNAL_FINGERPRINT)
print('Six class mapping:',CLASS_TO_ID)
print('External per-class counts:',external_df['class'].value_counts().sort_index().to_dict())

Stage 2 manifest verified: 1956 / 487 / 463. SHA-256: 9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792
Stage 3 frozen manifest verified: 4931 Fingerprint: 0761bdca0f92aea4efd8e5945e2954bf3d6958f41644e8699766839047ddd9be
Six class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
External per-class counts: {'cardboard': 981, 'glass': 1269, 'metal': 497, 'paper': 747, 'plastic': 1107, 'trash': 330}


## 4. Automatic dataset loading (KaggleHub native + direct-download fallback)

The public original dataset and the fixed V2 **version 12** dataset are loaded automatically. The notebook first reuses a recognized existing Kaggle mount, then attempts `kagglehub.dataset_download`; if Kaggle rejects datasource attachment with a backend error, it retries KaggleHub in direct HTTP download mode. No silent fallback to an unverified V2 version occurs. A Kaggle-side service outage or disabled Internet may still require attaching the same source through **Add Input**. All dataset contents are independently checked against the manifests in the following section.


In [4]:
# Section 4: resilient automatic dataset loading, adapted from the working
# Stage 3 Section 2 procedure. No alterations to checkpoints or manifests.
ORIGINAL_HANDLE = 'saumyamohandas/garbage-classification-image-dataset'
EXTERNAL_HANDLE = 'sumn2u/garbage-classification-v2/versions/12'

# KaggleHub's Kaggle-native resolver *attaches* the dataset by a backend POST.
# A backend POST failure is not proof the dataset URL or contents are invalid.
# Work around this by using the KaggleHub HTTP resolver on retry.
def _mounted_dataset_candidates(kind):
    if kind == 'original':
        return [
            Path('/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset'),
            Path('/kaggle/input/garbage-classification-image-dataset'),
        ]
    if kind == 'external':
        return [
            Path('/kaggle/input/datasets/sumn2u/garbage-classification-v2'),
            Path('/kaggle/input/garbage-classification-v2'),
        ]
    raise ValueError(f'Unknown dataset kind: {kind}')

def _has_plausible_dataset_content(root, kind):
    # This is only a cheap source-discovery test, NOT scientific validation.
    # The next section verifies actual SHA-256 bytes of every locked image.
    if not root.is_dir():
        return False
    if kind == 'original':
        for base in (root, root / 'dataset'):
            if (base / 'Training').is_dir() and (base / 'Testing').is_dir():
                return True
        # The original dataset can also be distributed as zipped source images.
        return any(root.glob('*.zip'))
    # External dataset can have extra wrapper folders. Detect a complete
    # six-class directory without relying on a specific wrapper-folder name.
    required = set(CLASSES)
    for base in (root, *(p for p in root.iterdir() if p.is_dir())):
        names = {p.name.lower() for p in base.iterdir() if p.is_dir()}
        if required.issubset(names):
            return True
    return any(root.glob('*.zip'))

def auto_resolve_public_dataset(handle, kind):
    # 1. Reuse a clearly identified already-attached Kaggle mount.
    for mount in _mounted_dataset_candidates(kind):
        if _has_plausible_dataset_content(mount, kind):
            print(f'[{kind}] Existing Kaggle input discovered: {mount}')
            return mount, 'existing Kaggle input'

    # 2. Use KaggleHub's normal native resolver, as in Section 2.
    failures = []
    try:
        print(f'[{kind}] KaggleHub automatic attach: {handle}', flush=True)
        p = Path(kagglehub.dataset_download(handle))
        if p.is_dir():
            print(f'[{kind}] KaggleHub attached: {p}')
            return p, 'kagglehub native attach'
        failures.append(f'Native resolver returned an invalid path: {p}')
    except Exception as exc:
        failures.append(f'Native attach {type(exc).__name__}: {str(exc)[:500]}')
        print(f'[{kind}] Native Kaggle attachment failed: {type(exc).__name__}', flush=True)

    # 3. KaggleHub *direct download* fallback. The documented setting prevents
    # kaggle_cache_resolver from making AttachDatasourceUsingJwtRequest,
    # allowing the ordinary KaggleHub HTTP download resolver to be tried.
    # Internet access is required if the dataset is not in the shared cache.
    key = 'DISABLE_KAGGLE_CACHE'
    original_env = os.environ.get(key)
    try:
        os.environ[key] = '1'
        print(f'[{kind}] Retrying automatically with KaggleHub direct HTTP download...',flush=True)
        p = Path(kagglehub.dataset_download(handle))
        if p.is_dir():
            print(f'[{kind}] KaggleHub direct download: {p}', flush=True)
            return p, 'kagglehub direct download (attachment bypass)'
        failures.append(f'Direct resolver returned an invalid path: {p}')
    except Exception as exc:
        failures.append(f'Direct download {type(exc).__name__}: {str(exc)[:500]}')
    finally:
        if original_env is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = original_env

    # The notebook must NEVER silently substitute another dataset/version.
    detail = '\n'.join(f' - {x}' for x in failures)
    raise RuntimeError(
        f'[{kind}] Dataset could not be resolved automatically: {handle}\n'
        f'{detail}\n'
        'Please check Kaggle notebook Internet access and public-dataset availability. '
        'If KaggleHub services are unavailable, attach the *same* public dataset '
        'via Add Input and rerun. Do not replace V2 version 12 or change the locked manifest.'
    )

print('Resolving original dataset:', ORIGINAL_HANDLE)
original_download, original_download_method = auto_resolve_public_dataset(ORIGINAL_HANDLE, 'original')
print('Resolving external V2 dataset:', EXTERNAL_HANDLE)
external_download, external_download_method = auto_resolve_public_dataset(EXTERNAL_HANDLE, 'external')
assert original_download.is_dir() and external_download.is_dir()
print('AUTOMATIC DATASET RESOLUTION SUCCEEDED')
print('Original:', original_download, 'via', original_download_method)
print('External:', external_download, 'via', external_download_method)


Resolving original dataset: saumyamohandas/garbage-classification-image-dataset
[original] KaggleHub automatic attach: saumyamohandas/garbage-classification-image-dataset
[original] Native Kaggle attachment failed: BackendError
[original] Retrying automatically with KaggleHub direct HTTP download...


100%|██████████| 43.4M/43.4M [00:00<00:00, 106MB/s] 

Extracting files...


[original] KaggleHub direct download: /root/.cache/kagglehub/datasets/saumyamohandas/garbage-classification-image-dataset/versions/1
Resolving external V2 dataset: sumn2u/garbage-classification-v2/versions/12
[external] KaggleHub automatic attach: sumn2u/garbage-classification-v2/versions/12
[external] Native Kaggle attachment failed: BackendError
[external] Retrying automatically with KaggleHub direct HTTP download...


100%|██████████| 1.07G/1.07G [00:05<00:00, 202MB/s] 

Extracting files...


[external] KaggleHub direct download: /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12
AUTOMATIC DATASET RESOLUTION SUCCEEDED
Original: /root/.cache/kagglehub/datasets/saumyamohandas/garbage-classification-image-dataset/versions/1 via kagglehub direct download (attachment bypass)
External: /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12 via kagglehub direct download (attachment bypass)


## 5. Resolve original and external image folders; verify every hash

This step checks all 2,906 entries of the original Stage 2 split manifest and all 4,931 entries of the external manifest against actual image bytes. Paths that do not resolve, wrong labels, mismatched checksums, or incorrect V2 image counts **stop execution**.

In [5]:
IMG_EXT={'.jpg','.jpeg','.png','.webp','.bmp'}

def safe_parts(relative):
    p=PurePosixPath(str(relative).replace('\\','/'))
    if p.is_absolute() or not p.parts or any(s in ('','.','..') for s in p.parts):
        raise ValueError(f'Unsafe image relative path: {relative!r}')
    return p.parts

def extract_original_archives(root):
    roots=[root]
    archives=sorted(root.rglob('*.zip'))
    for n,archive in enumerate(archives):
        destination=OUTPUT/'original_source_extracted'/f'{n:03d}_{archive.stem}'
        marker=destination/'.extract_complete'
        if not marker.exists():
            destination.mkdir(parents=True,exist_ok=True)
            with zipfile.ZipFile(archive) as z:
                for entry in z.infolist():
                    relative=PurePosixPath(entry.filename.replace('\\','/'))
                    if relative.is_absolute() or '..' in relative.parts:
                        raise RuntimeError(f'Unsafe archive member: {entry.filename}')
                    if not (destination/entry.filename).resolve().is_relative_to(destination.resolve()):
                        raise RuntimeError('Unsafe original archive member path')
                z.extractall(destination)
            marker.touch()
        roots.append(destination)
    return roots

def original_dataset_parents(root):
    found=[]
    for base in extract_original_archives(root):
        for parent in [base,*sorted((p for p in base.rglob('*') if p.is_dir()),key=str)]:
            dirs={p.name.lower():p for p in parent.iterdir() if p.is_dir()}
            train=dirs.get('training')
            test=dirs.get('testing')
            if train and test and {p.name for p in train.iterdir() if p.is_dir()}==set(CLASSES):
                if {p.name for p in test.iterdir() if p.is_dir()}==set(CLASSES):
                    found.append(parent)
    return found

def resolve_original(stage2,download):
    candidates=original_dataset_parents(download)
    print('Original Training/Testing directory candidates:',len(candidates))
    if not candidates:
        raise FileNotFoundError('Cannot locate original Training/Testing class folders.')
    expected_rows=[(safe_parts(r.source_path),r.sha256,r['class'])
                   for _,r in stage2.iterrows()]
    for parent in candidates:
        try:
            resolved=[]
            for parts,digest,cls in expected_rows:
                image=parent.joinpath(*parts)
                if not image.is_file() or len(parts)<3 or parts[1]!=cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'Original hash mismatch: {image}')
                resolved.append(str(image))
            print('Original split validated against image pixels:',parent)
            return resolved,parent
        except (FileNotFoundError,ValueError) as exc:
            print('Original candidate rejected:',parent,'reason:',str(exc)[:180])
    raise RuntimeError('No original dataset directory matches all 2906 locked SHA-256 records.')

def external_class_roots(root):
    # Prefer the known v12 `original` subdirectory, but verify by membership and hashes.
    possibilities=[root,root/'original']
    possibilities += [p for p in root.rglob('*') if p.is_dir() and p.name=='original']
    possibilities += [p for p in root.iterdir() if p.is_dir()]
    roots=[]
    for p in possibilities:
        if p.is_dir() and all((p/c).is_dir() for c in CLASSES) and p not in roots:
            roots.append(p)
    return roots

def resolve_external(external,download):
    candidates=external_class_roots(download)
    print('External class-root candidates:',candidates)
    if not candidates: raise FileNotFoundError('Missing six V2 class directories.')
    expected_rows=[(safe_parts(r.external_relative_path),r.sha256,r['class'])
                   for _,r in external.iterrows()]
    for root in candidates:
        try:
            dirs=[p for p in root.iterdir() if p.is_dir()]
            image_counts={p.name:sum(x.is_file() and x.suffix.lower() in IMG_EXT for x in p.rglob('*')) for p in dirs}
            if len(image_counts)!=10 or sum(image_counts.values())!=12259:
                raise RuntimeError(f'Expected 10 V2 folders / 12259 images, found {image_counts}')
            if sum(image_counts[c] for c in CLASSES)!=7463:
                raise RuntimeError('Version 12 six-class total differs from the screened audit.')
            resolved=[]
            for parts,digest,cls in expected_rows:
                image=root.joinpath(*parts)
                if not image.is_file() or parts[0]!=cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'External hash mismatch: {image}')
                resolved.append(str(image))
            print('External images SHA-256 verified:',len(resolved),'root:',root)
            return resolved,root,image_counts
        except (FileNotFoundError,ValueError,RuntimeError) as exc:
            print('External candidate rejected:',root,'reason:',str(exc)[:180])
    raise RuntimeError('No external dataset root matches all frozen image hashes and version 12 counts.')

stage2_paths, original_image_root=resolve_original(stage2_df,original_download)
external_paths, external_image_root, v2_counts=resolve_external(external_df,external_download)
stage2_df=stage2_df.copy(); stage2_df['resolved_path']=stage2_paths
external_df=external_df.copy(); external_df['resolved_path']=external_paths
original_test_df=stage2_df.loc[stage2_df['split']=='Testing'].sort_values('source_path').reset_index(drop=True)
external_eval_df=external_df.sort_values('external_relative_path').reset_index(drop=True)
assert len(original_test_df)==463 and len(external_eval_df)==4931
assert not set(original_test_df.sha256).intersection(set(external_eval_df.sha256))
print('Original test images:',len(original_test_df))
print('External evaluated images:',len(external_eval_df))

Original Training/Testing directory candidates: 1
Original split validated against image pixels: /root/.cache/kagglehub/datasets/saumyamohandas/garbage-classification-image-dataset/versions/1/dataset
External class-root candidates: [PosixPath('/root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12/original'), PosixPath('/root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12/standardized_384'), PosixPath('/root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12/standardized_256')]
External images SHA-256 verified: 4931 root: /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12/original
Original test images: 463
External evaluated images: 4931


## 6. Restore the exact Stage 2 REMNET ensemble and preprocessing

Stage 2 `Remnet_Accu_s2(3).ipynb` uses **three unwrapped modules**:
- `torchvision.models.mobilenet_v2(weights=None)`, replacing `classifier[1]` with six logits;
- `torchvision.models.resnet50(weights=None)`, replacing `fc` with six logits;
- `timm.create_model('efficientnet_lite0', pretrained=False, num_classes=6)`.

Each checkpoint is `{'state_dict', 'seed', 'split_sha256', 'model', ...}` and must strictly match its architecture, seed and split. Softmax probabilities are averaged equally. There is no training and no single-model substitution.


In [6]:
TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((484,484)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3,(0.5,)*3),
])

class LockedImageDataset(Dataset):
    def __init__(self, df, kind):
        self.df = df.reset_index(drop=True)
        self.path_col = 'source_path' if kind == 'original' else 'external_relative_path'
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row['resolved_path']) as im:
            image = TEST_TRANSFORM(im.convert('RGB'))
        return image, int(CLASS_TO_ID[row['class']]), i

def locked_loader(df,kind):
    return DataLoader(LockedImageDataset(df,kind),batch_size=BATCH_SIZE,
                      shuffle=False,num_workers=0,pin_memory=(DEVICE.type=='cuda'))

def construct_constituent(name):
    n = len(CLASSES)
    if name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == 'ResNet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == 'EfficientNet-Lite0':
        m = timm.create_model('efficientnet_lite0', pretrained=False, num_classes=n)
    else:
        raise ValueError(name)
    return m

def restore_checkpoint(seed):
    ensemble, metadata, digests = {}, {}, {}
    for name in CONSTITUENTS:
        path = CHECKPOINTS[seed][name]
        obj = torch.load(path,map_location='cpu',weights_only=True)
        assert isinstance(obj,dict) and 'state_dict' in obj, f'Unsupported Stage 2 checkpoint: {path}'
        assert str(obj.get('model')) == name, f'Constituent mismatch in {path.name}: {obj.get("model")}'
        assert int(obj.get('seed')) == seed, f'Seed mismatch in {path.name}'
        assert obj.get('split_sha256') == SPLIT_FINGERPRINT, f'Split fingerprint mismatch in {path.name}'
        m = construct_constituent(name)
        m.load_state_dict(obj['state_dict'],strict=True)
        m.to(DEVICE).eval()
        for param in m.parameters(): param.requires_grad_(False)
        ensemble[name],metadata[name],digests[name] = m,obj,file_sha256(path)
    return ensemble,metadata,digests

def infer(ensemble,df,kind):
    labels, probabilities, positions = [],[],[]
    with torch.inference_mode():
        for images,y,index in locked_loader(df,kind):
            images=images.to(DEVICE,non_blocking=True)
            # Exact Stage 2 REMNET rule: arithmetic mean of three softmax vectors.
            stacked = torch.stack([torch.softmax(ensemble[name](images),dim=1)
                                   for name in CONSTITUENTS],dim=0)
            prob=stacked.mean(dim=0)
            probabilities.append(prob.cpu().numpy().astype(np.float64))
            labels.append(y.numpy()); positions.append(index.numpy())
    labels=np.concatenate(labels).astype(np.int64)
    probabilities=np.concatenate(probabilities)
    positions=np.concatenate(positions)
    assert np.array_equal(positions,np.arange(len(df))), 'Image order changed during inference.'
    assert np.array_equal(labels,df['class'].map(CLASS_TO_ID).to_numpy()), 'Label order mismatch.'
    assert probabilities.shape==(len(df),6) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1),1,atol=2e-6)
    return labels,probabilities

_smoke, _smoke_meta, _smoke_hash = restore_checkpoint(SEEDS[0])
with torch.inference_mode():
    batch=torch.zeros(1,3,484,484,device=DEVICE)
    assert all(tuple(_smoke[name](batch).shape)==(1,6) for name in CONSTITUENTS)
print('REMNET: all three exact Stage 2 constituent architectures strictly loaded; six-logit smoke test PASS')
del _smoke,_smoke_meta,_smoke_hash,batch
if DEVICE.type=='cuda':torch.cuda.empty_cache()


REMNET: all three exact Stage 2 constituent architectures strictly loaded; six-logit smoke test PASS


## 7. Historical Stage 2 REMNET prediction matching (if available)

Looks for **REMNET ensemble** probabilities (`REMNET_seed*_test_probabilities.csv`), not standalone ResNet50 predictions. Missing individual-image reference files are recorded explicitly rather than silently substituted. They are never used for model selection.


In [7]:
def reference_for_seed(seed):
    filename=f'REMNET_seed{seed}_test_probabilities.csv'
    roots=[Path(HISTORICAL_REFERENCE_DIR)] if HISTORICAL_REFERENCE_DIR else [RESULTS_ROOT]
    hits=[p for root in roots if root.is_dir() for p in root.rglob(filename) if p.is_file()]
    if len(hits)>1: raise RuntimeError(f'Multiple historical references for seed {seed}: {hits}')
    return hits[0] if hits else None

def verify_historical_predictions(seed,df,predicted,probs):
    path=reference_for_seed(seed)
    if path is None:
        if REQUIRE_HISTORICAL_REFERENCE:
            raise RuntimeError(f'Historical individual REMNET reference unavailable for seed {seed}')
        return 'HISTORICAL_REFERENCE_UNAVAILABLE',None
    ref=pd.read_csv(path,dtype={'source_path':str})
    assert len(ref)==len(df)==463 and ref['source_path'].is_unique
    assert {'source_path','predicted_class_id'} <= set(ref.columns)
    aligned=ref.set_index('source_path').loc[df['source_path'].tolist()]
    ref_pred=aligned['predicted_class_id'].astype(int).to_numpy()
    if not np.array_equal(ref_pred,predicted):
        raise RuntimeError(f'Original Stage 2 prediction mismatch for REMNET seed {seed}')
    prob_columns=[f'prob_{c}' for c in CLASSES]
    if all(k in aligned.columns for k in prob_columns):
        expected=aligned[prob_columns].to_numpy(dtype=np.float64)
        if not np.allclose(expected,probs,atol=3e-5,rtol=3e-5):
            raise RuntimeError(f'Original Stage 2 probability mismatch for seed {seed}')
        return 'HISTORICAL_PREDICTIONS_AND_PROBABILITIES_MATCHED',str(path)
    return 'HISTORICAL_PREDICTED_LABELS_MATCHED',str(path)

## 8. Metrics, predictions, and confusion-matrix writers

Every metric is computed from the same fixed image ordering. Precision, recall and F1 use **macro averaging across all six classes** (`zero_division=0`). All CSV metric fields are stored as percentages.

In [8]:
METRICS=('accuracy','macro_precision','macro_recall','macro_f1')
PRED_DIR=OUTPUT/'Predictions'; PRED_DIR.mkdir(exist_ok=True)
CM_DIR=OUTPUT/'Confusion_Matrices'; CM_DIR.mkdir(exist_ok=True)

def measure(seed,kind,df,y_true,prob,checkpoint_sha,verification_status):
    y_pred=prob.argmax(axis=1).astype(int)
    precision,recall,f1,_=precision_recall_fscore_support(
        y_true,y_pred,labels=list(range(6)),average='macro',zero_division=0)
    scores={'model':MODEL_NAME,'seed':seed,'dataset':kind,'n_images':len(df),
            'accuracy':100*accuracy_score(y_true,y_pred),
            'macro_precision':100*precision,
            'macro_recall':100*recall,'macro_f1':100*f1,
            'checkpoint_sha256':checkpoint_sha,
            'original_test_historical_status':verification_status,
            'stage2_manifest_fingerprint':SPLIT_FINGERPRINT,
            'external_manifest_fingerprint':EXTERNAL_FINGERPRINT}
    return scores,y_pred

def save_predictions(seed,kind,df,y_true,y_pred,prob,checkpoint_sha):
    relative='source_path' if kind=='Original463' else 'external_relative_path'
    out=pd.DataFrame({
        'model':MODEL_NAME,'seed':seed,'dataset':kind,
        'image_relative_path':df[relative].astype(str).tolist(),
        'image_sha256':df['sha256'].tolist(),
        'true_class_id':y_true,
        'true_class':[CLASSES[i] for i in y_true],
        'predicted_class_id':y_pred,
        'predicted_class':[CLASSES[i] for i in y_pred],
        'prediction_confidence':prob.max(axis=1),
        'checkpoint_sha256':checkpoint_sha})
    for i,c in enumerate(CLASSES): out[f'prob_{c}']=prob[:,i]
    assert len(out)==len(df) and out['image_relative_path'].is_unique
    destination=PRED_DIR/f'REMNET_seed{seed}_{kind}_probabilities.csv'
    out.to_csv(destination,index=False,float_format='%.10g')
    return destination

def save_per_class(seed,kind,y_true,y_pred):
    values=classification_report(y_true,y_pred,labels=list(range(6)),target_names=list(CLASSES),
                                 output_dict=True,zero_division=0)
    records=[]
    for cls in CLASSES:
        r=values[cls]
        records.append({'model':MODEL_NAME,'seed':seed,'dataset':kind,'class':cls,
                        'output_index':CLASS_TO_ID[cls],'support':int(r['support']),
                        'precision_pct':100*r['precision'],
                        'recall_pct':100*r['recall'],'f1_pct':100*r['f1-score']})
    path=OUTPUT/f'REMNET_seed{seed}_{kind}_per_class.csv'
    pd.DataFrame(records).to_csv(path,index=False,float_format='%.10f')
    return path

def save_confusion(seed,kind,y_true,y_pred):
    cm=confusion_matrix(y_true,y_pred,labels=list(range(6)))
    path=CM_DIR/f'REMNET_seed{seed}_{kind}_confusion.csv'
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(path)
    fig,ax=plt.subplots(figsize=(7,6))
    ax.imshow(cm,cmap='Blues')
    ax.set_xticks(range(6),labels=CLASSES,rotation=45,ha='right')
    ax.set_yticks(range(6),labels=CLASSES)
    ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
    ax.set_title(f'REMNET | seed {seed} | {kind} | n={len(y_true)}')
    threshold=cm.max()/2
    for i in range(6):
        for j in range(6):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                    color='white' if cm[i,j]>threshold else 'black',fontsize=9)
    fig.tight_layout()
    fig.savefig(CM_DIR/f'REMNET_seed{seed}_{kind}_confusion.png',dpi=160)
    plt.close(fig)
    return path

## 9. Run five seed-matched REMNET ensembles on original 463 and external 4,931 images

For each seed, load exactly three frozen Stage 2 REMNET constituents; verify D1 repeatability and optional Stage 2 historical predictions; then infer on the same locked D2 cohort. No model training, tuning or adaptation occurs.


In [9]:
per_seed=[]
checkpoint_registry=[]
original_reference_statuses=[]
for seed in SEEDS:
    print(f'\n=== REMNET ENSEMBLE SEED {seed} ===',flush=True)
    ensemble, metadata, hashes = restore_checkpoint(seed)
    for name in CONSTITUENTS:
        checkpoint_registry.append({
            'model':MODEL_NAME,'seed':seed,'constituent':name,
            'checkpoint_path':str(CHECKPOINTS[seed][name]),
            'checkpoint_sha256':hashes[name],
            'checkpoint_split_sha256':metadata[name]['split_sha256'],
            'checkpoint_best_epoch':metadata[name].get('best_epoch'),
            'framework':('timm' if name=='EfficientNet-Lite0' else 'torchvision'),
            'num_output_classes':6,'input_size':'32x32',
            'normalization_mean':'0.5,0.5,0.5','normalization_std':'0.5,0.5,0.5',
            'external_manifest_fingerprint':EXTERNAL_FINGERPRINT})
    ensemble_sha=json.dumps(hashes,sort_keys=True)  # exact three-source provenance in each prediction CSV
    y_true_orig,p_orig=infer(ensemble,original_test_df,'original')
    y_true_repeat,p_repeat=infer(ensemble,original_test_df,'original')
    assert np.array_equal(y_true_orig,y_true_repeat)
    assert np.allclose(p_orig,p_repeat,atol=2e-6,rtol=2e-6),f'Original test not repeatable: seed {seed}'
    historical_status,historical_file=verify_historical_predictions(seed,original_test_df,p_orig.argmax(1),p_orig)
    original_reference_statuses.append({'seed':seed,'status':historical_status,'reference_file':historical_file or ''})
    orig_score,pred_orig=measure(seed,'Original463',original_test_df,y_true_orig,p_orig,ensemble_sha,historical_status)
    per_seed.append(orig_score)
    save_predictions(seed,'Original463',original_test_df,y_true_orig,pred_orig,p_orig,ensemble_sha)
    save_per_class(seed,'Original463',y_true_orig,pred_orig)
    save_confusion(seed,'Original463',y_true_orig,pred_orig)
    print('Original n:',len(y_true_orig),'| Accuracy:',round(orig_score['accuracy'],4),
          '| Macro-F1:',round(orig_score['macro_f1'],4))
    y_true_ext,p_ext=infer(ensemble,external_eval_df,'external')
    ext_score,pred_ext=measure(seed,'External4931',external_eval_df,y_true_ext,p_ext,ensemble_sha,historical_status)
    per_seed.append(ext_score)
    save_predictions(seed,'External4931',external_eval_df,y_true_ext,pred_ext,p_ext,ensemble_sha)
    save_per_class(seed,'External4931',y_true_ext,pred_ext)
    save_confusion(seed,'External4931',y_true_ext,pred_ext)
    print('External n:',len(y_true_ext),'| Accuracy:',round(ext_score['accuracy'],4),
          '| Macro-F1:',round(ext_score['macro_f1'],4))
    del ensemble
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

results_df=pd.DataFrame(per_seed)
registry_df=pd.DataFrame(checkpoint_registry)
results_df.to_csv(OUTPUT/'REMNET_All_Seeds_Original_vs_External.csv',index=False,float_format='%.10f')
registry_df.to_csv(OUTPUT/'REMNET_Checkpoint_Registry.csv',index=False)
pd.DataFrame(original_reference_statuses).to_csv(OUTPUT/'REMNET_Historical_Reference_Status.csv',index=False)
assert len(registry_df)==15 and len(results_df)==10
assert results_df.groupby('dataset').size().to_dict()=={'Original463':5,'External4931':5}
print('\nPER-SEED REMNET RESULTS:')
print(results_df[['seed','dataset','n_images',*METRICS]].to_string(index=False,float_format=lambda v:f'{v:.4f}'))



=== REMNET ENSEMBLE SEED 7 ===
Original n: 463 | Accuracy: 94.6004 | Macro-F1: 93.7914
External n: 4931 | Accuracy: 63.4962 | Macro-F1: 59.3616

=== REMNET ENSEMBLE SEED 49 ===
Original n: 463 | Accuracy: 94.8164 | Macro-F1: 93.9637
External n: 4931 | Accuracy: 62.462 | Macro-F1: 58.0797

=== REMNET ENSEMBLE SEED 112 ===
Original n: 463 | Accuracy: 93.7365 | Macro-F1: 92.4941
External n: 4931 | Accuracy: 62.7256 | Macro-F1: 58.7502

=== REMNET ENSEMBLE SEED 345 ===
Original n: 463 | Accuracy: 94.1685 | Macro-F1: 93.1294
External n: 4931 | Accuracy: 62.7053 | Macro-F1: 59.0225

=== REMNET ENSEMBLE SEED 476 ===
Original n: 463 | Accuracy: 93.3045 | Macro-F1: 92.0023
External n: 4931 | Accuracy: 61.0627 | Macro-F1: 57.227

PER-SEED REMNET RESULTS:
 seed      dataset  n_images  accuracy  macro_precision  macro_recall  macro_f1
    7  Original463       463   94.6004          94.6182       93.3759   93.7914
    7 External4931      4931   63.4962          59.0056       61.9662   59.3616
   4

## 10. Mean ± sample standard deviation: REMNET original versus external

Exactly the same five-seed summary and generalization-gap calculation as the supplied standalone ResNet50 Stage 3 notebook, now on seed-matched REMNET probability ensembles.


In [10]:
summary=[]
for metric in METRICS:
    old=results_df.loc[results_df.dataset=='Original463'].set_index('seed').loc[list(SEEDS),metric]
    new=results_df.loc[results_df.dataset=='External4931'].set_index('seed').loc[list(SEEDS),metric]
    gap=new-old
    summary.append({
        'model':MODEL_NAME,'metric':metric,'n_seeds':5,
        'original_test_images':463,'external_images':4931,
        'original_mean_pct':old.mean(),'original_sample_sd_pct':old.std(ddof=1),
        'external_mean_pct':new.mean(),'external_sample_sd_pct':new.std(ddof=1),
        'external_minus_original_mean_pp':gap.mean(),
        'seedwise_gap_sample_sd_pp':gap.std(ddof=1)})
summary_df=pd.DataFrame(summary)
summary_df.to_csv(OUTPUT/'REMNET_Five_Seed_Mean_SD_Original_vs_External.csv',
                  index=False,float_format='%.10f')
print('REMNET — ORIGINAL 463 vs EXTERNAL 4931')
for row in summary_df.itertuples(index=False):
    print(f'{row.metric:17s}: {row.original_mean_pct:.2f} +/- {row.original_sample_sd_pct:.2f}%'
          f'  ->  {row.external_mean_pct:.2f} +/- {row.external_sample_sd_pct:.2f}%'
          f'   (gap {row.external_minus_original_mean_pp:+.2f} pp)')

# Historical Stage 2 aggregate and optional per-seed consistency checks.
# These verify the *original* 463-image test results; they never drive selection.
historical_checks=[]

def pct_from_historical(value):
    v=float(value)
    if not np.isfinite(v): raise ValueError('Non-finite historical metric')
    return 100.0*v if abs(v)<=1.000001 else v

if HISTORICAL_PER_SEED is not None:
    prior=pd.read_csv(HISTORICAL_PER_SEED)
    assert {'seed', *METRICS} <= set(prior.columns), f'Unexpected per-seed historical CSV columns: {prior.columns.tolist()}'
    assert len(prior)==5 and set(prior.seed.astype(int))==set(SEEDS)
    original=results_df[results_df.dataset=='Original463'].set_index('seed')
    for seed in SEEDS:
        row=prior.set_index('seed').loc[seed]
        for metric in METRICS:
            expected=pct_from_historical(row[metric]); observed=float(original.loc[seed, metric])
            difference=observed-expected
            historical_checks.append({'reference_type':'per_seed','seed':seed,'metric':metric,
                                      'historical_pct':expected,'reproduced_pct':observed,
                                      'difference_pp':difference,
                                      'within_0p15_percentage_points':abs(difference)<=0.15})

if HISTORICAL_SUMMARY is not None:
    prior=pd.read_csv(HISTORICAL_SUMMARY)
    assert {'metric','mean','sd'} <= set(prior.columns), f'Unexpected summary historical CSV columns: {prior.columns.tolist()}'
    assert set(METRICS) <= set(prior.metric)
    for metric in METRICS:
        hist=prior.set_index('metric').loc[metric]
        if isinstance(hist,pd.DataFrame):raise RuntimeError(f'Duplicate historical summary metric: {metric}')
        original=results_df[results_df.dataset=='Original463'][metric]
        for statistic,key,observed in [('mean','mean',original.mean()),('sd','sd',original.std(ddof=1))]:
            expected=pct_from_historical(hist[key]); difference=float(observed)-expected
            historical_checks.append({'reference_type':'five_seed_'+statistic,
                                      'seed':'all','metric':metric,
                                      'historical_pct':expected,'reproduced_pct':float(observed),
                                      'difference_pp':difference,
                                      'within_0p15_percentage_points':abs(difference)<=0.15})

historical_check_df=pd.DataFrame(historical_checks)
if not historical_check_df.empty:
    historical_check_df.to_csv(OUTPUT/'REMNET_Stage2_Historical_Performance_Verification.csv',index=False)
    failures=historical_check_df[~historical_check_df.within_0p15_percentage_points]
    print('Original-test Stage 2 historical metric comparisons:',len(historical_check_df),
          '| within 0.15 percentage points:',len(historical_check_df)-len(failures))
    if not failures.empty:
        print('WARNING: Historical original-test performance differs beyond 0.15 pp. Inspect before reporting:')
        print(failures[['reference_type','seed','metric','historical_pct','reproduced_pct','difference_pp']].to_string(index=False))
else:
    print('No original Stage 2 metric comparison CSV found; per-image verification remains independent.')


REMNET — ORIGINAL 463 vs EXTERNAL 4931
accuracy         : 94.13 +/- 0.62%  ->  62.49 +/- 0.89%   (gap -31.63 pp)
macro_precision  : 94.22 +/- 0.63%  ->  58.33 +/- 0.92%   (gap -35.89 pp)
macro_recall     : 92.53 +/- 0.88%  ->  61.24 +/- 0.62%   (gap -31.29 pp)
macro_f1         : 93.08 +/- 0.84%  ->  58.49 +/- 0.85%   (gap -34.59 pp)
Original-test Stage 2 historical metric comparisons: 20 | within 0.15 percentage points: 20


## 11. REMNET provenance and outputs

Exports all five-seed predictions and probabilities, per-class metrics, confusion matrices, mean ± sample SD, 15 constituent SHA-256s, and the original/external comparison, with no training or data changes.


In [11]:
verified_historical=sum(x['status'].startswith('HISTORICAL_PREDICT') for x in original_reference_statuses)
report={
    'experiment':'REMNET Stage 3 Section 3+4 — REMNET equal-weight ensemble inference',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_notebook':'Remnet_Accu_s2(3).ipynb',
    'checkpoint_source':'Stage 2 REMNET three seed-matched best-validation-loss constituent checkpoints',
    'checkpoint_dataset_input':str(INPUT_ROOT),'checkpoint_seeds':list(SEEDS),
    'stage2_original_dataset':ORIGINAL_HANDLE,
    'stage3_external_dataset':EXTERNAL_HANDLE,
    'stage3_external_dataset_version':EXPECTED_V2_VERSION,
    'original_download_method':original_download_method,
    'external_download_method':external_download_method,
    'original_resolved_path':str(original_download),
    'external_resolved_path':str(external_download),
    'original_test_count':463,'external_evaluation_count':4931,
    'stage2_split_fingerprint_sha256':SPLIT_FINGERPRINT,
    'external_locked_manifest_fingerprint_sha256':EXTERNAL_FINGERPRINT,
    'class_to_idx':CLASS_TO_ID,
    'original_test_transform':'RGB,Resize(32,32),ToTensor,Normalize(mean=.5x3,std=.5x3)',
    'external_transform':'identical to original test transform',
    'model_constructor':'REMNET_Model wrapper around torchvision.models.resnet50(weights=None); model.fc Linear(2048,6)',
    'model_loading':'three strict-loaded seed-matched constituent state_dicts; mean of three per-class softmax vectors; eval; inference_mode',
    'training_or_adaptation_performed':False,
    'checkpoint_selection_on_external_dataset':False,
    'external_subset_shared_across_models':'Section2_Final_Automated_Screened_Six_Class_Manifest.csv',
    'original_checkpoint_reference_status_per_seed':original_reference_statuses,
    'historical_stage2_per_seed_results_file':str(HISTORICAL_PER_SEED) if HISTORICAL_PER_SEED else None,
    'historical_stage2_summary_file':str(HISTORICAL_SUMMARY) if HISTORICAL_SUMMARY else None,
    'historical_performance_rows_compared':len(historical_checks),
    'historical_performance_mismatches_0p15pp':int(sum(not x['within_0p15_percentage_points'] for x in historical_checks)),
    'historical_individual_REMNET_reference_confirmed_seeds':verified_historical,
    'ensemble_constituents':list(CONSTITUENTS),
    'ensemble_rule':'equal-weight arithmetic mean of three per-class Softmax probability vectors',
    'original_463_test_sha256_verified':True,
    'external_4931_test_sha256_verified':True,
    'same_checkpoints_for_future_energy_inference':'REMNET_Checkpoint_Registry.csv: match all three constituent SHA-256s per seed',
    'torch':torch.__version__,
    'torchvision':__import__('torchvision').__version__,
    'device':str(DEVICE),
    'limitations':[ 
        'Automated conservative screen does not demonstrate independent dataset acquisition.',
        'Historical REMNET original-test prediction files may be absent. Aggregate Stage 2 metric agreement does not establish exact per-image historical prediction agreement.',
        'The datasets have different sample sizes and collection/provenance; differences are descriptive generalization gaps, not causal effects.'
    ]}
(OUTPUT/'REMNET_External_Validation_Configuration.json').write_text(
    json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')

archive=OUTPUT/'REMNET_Stage3_REMNET_Section3_4_Outputs.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as outzip:
    for f in sorted(OUTPUT.rglob('*')):
        if f.is_file() and f!=archive and 'original_source_extracted' not in f.parts:
            outzip.write(f,f.relative_to(OUTPUT))
print('OUTPUTS SAVED:',OUTPUT)
print('ZIP:',archive)
print('Original prediction reference matching confirmed for',verified_historical,'of five seeds.')
print('Historical original-test metric checks:',len(historical_checks))
if verified_historical<5:
    print('LIMITATION: Historical per-seed REMNET predictions unavailable for some/all seeds.')
    print('Checkpoint identity, original test pixels and repeatability were independently verified; historical predictions were not invented.')
print('STAGE 3 STANDALONE REMNET CLASSIFICATION EVALUATION COMPLETE: True')

OUTPUTS SAVED: /kaggle/working/REMNET_Stage3_Ensemble_External_Validation
ZIP: /kaggle/working/REMNET_Stage3_Ensemble_External_Validation/REMNET_Stage3_REMNET_Section3_4_Outputs.zip
Original prediction reference matching confirmed for 0 of five seeds.
Historical original-test metric checks: 20
LIMITATION: Historical per-seed REMNET predictions unavailable for some/all seeds.
Checkpoint identity, original test pixels and repeatability were independently verified; historical predictions were not invented.
STAGE 3 STANDALONE REMNET CLASSIFICATION EVALUATION COMPLETE: True
